# Day 36: Simulate Draft-Target Speculative Decoding

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

Implements the complete speculative decoding algorithm with acceptance sampling. The draft model proposes K tokens; the target verifies all K+1 positions in one parallel pass. Rejection resamples from a corrected distribution to maintain exact target distribution.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Draft-Target Speculative Decoding Simulation


In [ ]:
import random

class TinyLanguageModel:
    """Simulates a language model with a fixed token distribution."""
    def __init__(self, vocab_size=100, quality=1.0, seed=42):
        np.random.seed(seed)
        self.vocab = vocab_size
        self.quality = quality  # 1.0 = perfect, 0.5 = random
        # Fixed 'true' distribution
        self.true_logits = np.random.randn(vocab_size)

    def next_token_probs(self, context):
        # Quality controls how close to 'true' distribution
        noise = np.random.randn(self.vocab) * (1 - self.quality) * 2
        logits = self.true_logits + noise
        exp_l = np.exp(logits - logits.max())
        return exp_l / exp_l.sum()

class SpeculativeDecoder:
    def __init__(self, target_model, draft_model, K=4):
        self.target = target_model
        self.draft = draft_model
        self.K = K

    def decode_step(self, context):
        """One speculative decode step. Returns (tokens_generated, accepted)."""
        # 1. Draft proposes K tokens
        draft_tokens = []
        draft_probs = []
        ctx = list(context)
        for _ in range(self.K):
            p_draft = self.draft.next_token_probs(ctx)
            tok = np.random.choice(len(p_draft), p=p_draft)
            draft_tokens.append(tok)
            draft_probs.append(p_draft[tok])
            ctx.append(tok)

        # 2. Target verifies all K+1 positions in parallel
        accepted = []
        for k, (tok, p_d) in enumerate(zip(draft_tokens, draft_probs)):
            p_target = self.target.next_token_probs(list(context) + draft_tokens[:k])
            accept_prob = min(1.0, p_target[tok] / (p_d + 1e-8))
            if np.random.random() < accept_prob:
                accepted.append(tok)
            else:
                # Resample from corrected distribution
                p_corrected = np.maximum(0, p_target - self.draft.next_token_probs(
                    list(context) + draft_tokens[:k]))
                if p_corrected.sum() > 0:
                    p_corrected /= p_corrected.sum()
                    bonus = np.random.choice(len(p_corrected), p=p_corrected)
                else:
                    bonus = np.argmax(p_target)
                accepted.append(bonus)
                break  # rejection ends the chain
        return accepted

    def benchmark(self, n_steps=500):
        context = [0, 1, 2, 3, 4]
        total_tokens = 0
        for _ in range(n_steps):
            accepted = self.decode_step(context)
            context.extend(accepted)
            total_tokens += len(accepted)
        mean_tokens_per_step = total_tokens / n_steps
        return mean_tokens_per_step

target = TinyLanguageModel(quality=1.0)
print('Speculative decoding: mean tokens/step vs draft quality')
for dq in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    draft = TinyLanguageModel(quality=dq, seed=99)
    spec = SpeculativeDecoder(target, draft, K=4)
    mean = spec.benchmark()
    speedup = mean  # vs 1 token per step for standard decoding
    print(f'  Draft quality={dq:.2f}: {mean:.2f} tokens/step ({speedup:.2f}x speedup)')


## 2. Acceptance Rate Distribution


In [ ]:
# Measure acceptance rate at each draft position
target = TinyLanguageModel(quality=1.0)
draft = TinyLanguageModel(quality=0.8, seed=99)
spec = SpeculativeDecoder(target, draft, K=8)

pos_accepts = [[] for _ in range(8)]
for _ in range(1000):
    context = list(range(5))
    ctx = list(context)
    for k in range(8):
        p_d = draft.next_token_probs(ctx)
        tok = np.random.choice(len(p_d), p=p_d)
        p_t = target.next_token_probs(ctx)
        accept_prob = min(1.0, p_t[tok] / (p_d[tok] + 1e-8))
        pos_accepts[k].append(accept_prob)
        ctx.append(tok)

plt.figure(figsize=(8,4))
plt.bar(range(1, 9), [np.mean(p) for p in pos_accepts])
plt.xlabel('Draft token position (k)')
plt.ylabel('Mean acceptance probability')
plt.title('Acceptance Rate by Draft Position\n(quality=0.8, independent per-position)')
plt.ylim(0, 1); plt.grid(True, axis='y')
plt.savefig('acceptance_by_position.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mean acceptance rate: {np.mean([np.mean(p) for p in pos_accepts]):.2f}')
print(f'Cumulative: {np.prod([np.mean(p) for p in pos_accepts]):.4f} (prob all 8 accepted)')


## Experiments

1. Try K=1,2,4,8,16. At what K does the benefit plateau for draft quality=0.8?
2. Implement tree-based speculation: draft generates a tree of K candidates at each position.
3. Simulate the overhead of running the draft model (cost_ratio=0.1) and compute real speedup.


## Key Takeaways

- See concept overview above for the key points.
- Day 36 complete.
